In [1]:
import pandas as pd
import os

# Cargar los datos desde la carpeta raw
# Usamos ../ para subir un nivel desde la carpeta notebooks y luego entrar a data/raw
path_raw = os.path.join('..', 'data', 'raw', 'ventas.csv')
df = pd.read_csv(path_raw)

print("Datos cargados exitosamente. Primeras filas:")
df.head()

Datos cargados exitosamente. Primeras filas:


,id,precio,fecha,nombre,ciudad,productos,email,id_vendedor
0,1,10000,01/09/2023,JUAN LOPEZ,Santiago,2.0,juan.lopez@gmail.com,0
1,2,$5000,01/10/2023,Roberto Alfredo,Concepcion,5.0,roberto.alfredo@gmail.com,0
2,3,NaN,02/10/2023,Carolina Herrera,Valdivia,5.0,carolina.herrera@gmail.com,1
3,4,6000,02/11/2023,Juan Lopez,Valparaiso,-6.0,juan.lopez@gmail.com,2
4,5,156000,04/10/2023,Juan Lopez,Santiago,1.0,juan.lopez@gmail.com,1


In [2]:
# Ver información básica de los datos y detectar nulos
print("--- Información de los datos ---")
print(df.info())

print("\n--- Conteo de valores nulos ---")
print(df.isnull().sum())

print("\n--- Valores únicos en 'ciudad' (para ver inconsistencias) ---")
print(df['ciudad'].unique())

--- Información de los datos ---
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           20 non-null     int64  
 1   precio       18 non-null     str    
 2   fecha        20 non-null     str    
 3   nombre       19 non-null     str    
 4   ciudad       18 non-null     str    
 5   productos    18 non-null     float64
 6   email        20 non-null     str    
 7   id_vendedor  20 non-null     int64  
dtypes: float64(1), int64(2), str(5)
memory usage: 2.5 KB
None

--- Conteo de valores nulos ---
id             0
precio         2
fecha          0
nombre         1
ciudad         2
productos      2
email          0
id_vendedor    0
dtype: int64

--- Valores únicos en 'ciudad' (para ver inconsistencias) ---
<ArrowStringArray>
[  'Santiago', 'Concepcion',   'Valdivia', 'Valparaiso', 'CONCEPCION',
          nan,   'sANTIAGO',  ' Valdivia']
Length: 8, dtyp

In [4]:
# 1. Limpiar la columna precio
# Quitamos el $ y convertimos a número. Lo que sea texto (como 'AAA') se volverá NaN (nulo)
df['precio_limpio'] = df['precio'].replace('[\$,]', '', regex=True)
df['precio_limpio'] = pd.to_numeric(df['precio_limpio'], errors='coerce')

# 2. Normalizar ciudades
# Todo a mayúsculas y quitamos espacios extra
df['ciudad_limpia'] = df['ciudad'].str.upper().str.strip()

print("Resultado de la primera limpieza:")
df[['id', 'precio', 'precio_limpio', 'ciudad', 'ciudad_limpia']].head(10)

Resultado de la primera limpieza:


<>:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
C:\Users\gabo\AppData\Local\Temp\ipykernel_4684\2688225455.py:3: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  df['precio_limpio'] = df['precio'].replace('[\$,]', '', regex=True)


,id,precio,precio_limpio,ciudad,ciudad_limpia
0,1,10000,10000.0,Santiago,SANTIAGO
1,2,$5000,5000.0,Concepcion,CONCEPCION
2,3,NaN,NaN,Valdivia,VALDIVIA
3,4,6000,6000.0,Valparaiso,VALPARAISO
4,5,156000,156000.0,Santiago,SANTIAGO
5,6,-45000,-45000.0,Santiago,SANTIAGO
6,7,84500,84500.0,CONCEPCION,CONCEPCION
7,8,120000,120000.0,Concepcion,CONCEPCION
8,9,$2000,2000.0,Valparaiso,VALPARAISO
9,10,6000,6000.0,Valparaiso,VALPARAISO


In [5]:
# 1. VALIDACIÓN SEMÁNTICA (Eliminar anomalías)
# Filtramos para quedarnos solo con precios mayores a 0 y que no sean nulos
df_validado = df[df['precio_limpio'] > 0].copy()

# También validamos que la cantidad de productos sea positiva
df_validado = df_validado[df_validado['productos'] > 0]

# 2. SEGURIDAD (Anonimización de datos sensibles - Ley 19.628)
# Vamos a ocultar parte del email para que no sea visible para todos
def anonimizar_email(email):
    if pd.isna(email): return "no_email@tienda.com"
    parte_usuario = email.split('@')[0]
    return f"{parte_usuario[:3]}***@dominio.com"

df_validado['email_anonimo'] = df_validado['email'].apply(anonimizar_email)

# 3. CARGA (Guardar el resultado en la carpeta processed)
path_processed = os.path.join('..', 'data', 'processed', 'ventas_limpias.csv')
df_validado.to_csv(path_processed, index=False)

print(f"Validación terminada. Filas originales: {len(df)} | Filas validadas: {len(df_validado)}")
print("\nPrimeras filas del dato final procesado:")
df_validado[['id', 'precio_limpio', 'ciudad_limpia', 'productos', 'email_anonimo']].head()

Validación terminada. Filas originales: 20 | Filas validadas: 11

Primeras filas del dato final procesado:


,id,precio_limpio,ciudad_limpia,productos,email_anonimo
0,1,10000.0,SANTIAGO,2.0,jua***@dominio.com
1,2,5000.0,CONCEPCION,5.0,rob***@dominio.com
4,5,156000.0,SANTIAGO,1.0,jua***@dominio.com
6,7,84500.0,CONCEPCION,2.0,ped***@dominio.com
7,8,120000.0,CONCEPCION,4.0,rob***@dominio.com
